In [1]:
from pathlib import Path
import zipfile

zip_files = list(Path("/content").glob("*.zip"))

if len(zip_files) != 1:
    print("ZIP files found:", [p.name for p in zip_files])
    raise ValueError("Upload one ZIP file, or specify its path.")

zip_path = zip_files[0]
extract_folder = Path("/content/housing_data")

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(extract_folder)

csv_files = [
    p for p in extract_folder.rglob("*.csv")
    if "__MACOSX" not in p.parts and not p.name.startswith("._")
]

print("Extracted to:", extract_folder)
print("Number of CSV files:", len(csv_files))

Extracted to: /content/housing_data
Number of CSV files: 4


In [3]:
from pathlib import Path
import pandas as pd

matches = list(Path("/content").rglob("train_prepared_24months.csv"))
if len(matches) != 1:
    raise ValueError(f"Expected one training CSV, found: {matches}")

sales = pd.read_csv(matches[0], dtype={"PostalCode": "string"})
dates = pd.to_datetime(sales["CloseDate"], errors="raise", utc=True)

train = sales.loc[
    (dates >= "2024-09-01") & (dates < "2026-06-01")
].copy()

validation = sales.loc[
    (dates >= "2026-06-01") & (dates < "2026-09-01")
].copy()

print(f"Training sales before cutoff: {len(train):,}")
print(f"June–August validation sales: {len(validation):,}")

Training sales before cutoff: 227,747
June–August validation sales: 37,828


In [4]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

price_cutoff = 17_000_000
train_screened = train.loc[train["ClosePrice"] <= price_cutoff].copy()

numeric = [
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "YearBuilt"
]
categories = ["City", "CountyOrParish", "PostalCode"]
flags = [
    f"{column}_{suffix}"
    for column in numeric
    for suffix in ["invalid_flag", "missing_flag"]
]
features = numeric + categories + flags

X_train = train_screened[features].copy()
X_valid = validation[features].copy()

for frame in [X_train, X_valid]:
    for flag in flags:
        frame[flag] = frame[flag].astype(str).str.lower().isin(["true", "1"])
    for column in numeric:
        frame[column] = frame[column].mask(frame[f"{column}_missing_flag"])

preprocessing = ColumnTransformer([
    ("numeric", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
    ("location", OneHotEncoder(handle_unknown="ignore"), categories),
    ("flags", "passthrough", flags),
])

model = make_pipeline(preprocessing, LinearRegression())
model.fit(X_train, train_screened["ClosePrice"])

predictions = model.predict(X_valid)
actual = validation["ClosePrice"]

print(f"Training sales excluded above $17M: {len(train) - len(train_screened):,}")
print(f"Training sales used: {len(train_screened):,}")
print(f"Validation sales scored: {len(actual):,}")
print(f"MAE:  ${mean_absolute_error(actual, predictions):,.0f}")
print(f"RMSE: ${np.sqrt(mean_squared_error(actual, predictions)):,.0f}")
print(f"R²:   {r2_score(actual, predictions):.4f}")

Training sales excluded above $17M: 245
Training sales used: 227,502
Validation sales scored: 37,828
MAE:  $307,174
RMSE: $890,219
R²:   0.6653


“I trained a linear regression model using sales of $17 million or less and evaluated it on all 37,828 sales from June through August 2026.”

In [5]:
from sklearn.base import clone

# Restore missing values using the same flags as in the screened fit.
X_train_full = train[features].copy()

for flag in flags:
    X_train_full[flag] = (
        X_train_full[flag].astype(str).str.lower().isin(["true", "1"])
    )

for column in numeric:
    X_train_full[column] = X_train_full[column].mask(
        X_train_full[f"{column}_missing_flag"]
    )

# Fit fresh preprocessing and regression on all training rows.
model_full = clone(model)
model_full.fit(X_train_full, train["ClosePrice"])
predictions_full = model_full.predict(X_valid)

comparison = pd.DataFrame([
    {
        "Model": name,
        "Training sales": count,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
    }
    for name, count, predicted in [
        ("All training sales", len(train), predictions_full),
        ("$17M training cutoff", len(train_screened), predictions),
    ]
])

display(comparison.style.format({
    "Training sales": "{:,}",
    "MAE": "${:,.0f}",
    "RMSE": "${:,.0f}",
    "R²": "{:.4f}",
}))

,Model,Training sales,MAE,RMSE,R²
0,All training sales,"227,747","$384,323","$923,397",0.6399
1,$17M training cutoff,"227,502","$307,174","$890,219",0.6653


In [6]:
from sklearn.base import clone

# Load September's prepared sales.
test_path = matches[0].with_name("test_prepared_202609.csv")
september = pd.read_csv(test_path, dtype={"PostalCode": "string"})

# Refit using both the earlier training months and the validation months.
refit_sales = pd.concat([train, validation], ignore_index=True)
final_train = refit_sales.loc[
    refit_sales["ClosePrice"] <= price_cutoff
].copy()

X_final_train = final_train[features].copy()
X_september = september[features].copy()

for frame in [X_final_train, X_september]:
    for flag in flags:
        frame[flag] = (
            frame[flag].astype(str).str.strip().str.lower().isin(["true", "1"])
        )
    for column in numeric:
        frame[column] = frame[column].mask(frame[f"{column}_missing_flag"])

final_linear = clone(model)
final_linear.fit(X_final_train, final_train["ClosePrice"])

september_actual = september["ClosePrice"]
september_pred = final_linear.predict(X_september)

print(f"Training sales excluded above $17M: {len(refit_sales) - len(final_train):,}")
print(f"Training sales used: {len(final_train):,}")
print(f"September sales scored: {len(september_actual):,}")
print(f"September MAE:  ${mean_absolute_error(september_actual, september_pred):,.0f}")
print(f"September RMSE: ${np.sqrt(mean_squared_error(september_actual, september_pred)):,.0f}")
print(f"September R²:   {r2_score(september_actual, september_pred):.4f}")
print(f"Negative September predictions: {(september_pred < 0).sum():,}")

Training sales excluded above $17M: 292
Training sales used: 265,283
September sales scored: 11,186
September MAE:  $308,208
September RMSE: $768,355
September R²:   0.7081
Negative September predictions: 103


“After validating the $17 million training cutoff on June–August 2026 sales, I refit Linear Regression on 265,283 sales from September 2024 through August 2026. I evaluated the refitted model on all 11,186 September 2026 sales.”

The biggest improvement was in MAE. On the same September homes, the average absolute error fell by about $72,753 per sale (19.1%). R² rose by 3.20 percentage points, and negative predictions fell by about half. The earlier June–August comparison also favored the cutoff, so the September results are consistent with what you saw in validation.